In [12]:
import os
from openai import OpenAI
from pageindex import PageIndexClient
from dotenv import load_dotenv

In [15]:
load_dotenv()

True

In [6]:
print(os.getenv("PAGEINDEX_API_KEY"))

17e374a486b44fe6be04e77c47f6d42f


In [7]:
pi_client = PageIndexClient(api_key=os.getenv("PAGEINDEX_API_KEY"))

In [11]:
result = pi_client.get_tree("pi-cmpxsacnz03a701qut3yzyphx", node_summary=True)
result

{'doc_id': 'pi-cmpxsacnz03a701qut3yzyphx',
 'status': 'completed',
 'retrieval_ready': True,
 'result': [{'title': 'Contents',
   'node_id': '0000',
   'page_index': 1,
   'summary': '# Contents\n\nMental health of older persons: a human right and priority for sustainable development 2\n\nDeterminants of mental health in later life 2\n\nPrevalence of mental health conditions among older persons in the UNECE region 5\n\nPolicy strategies to improve the mental health of older persons 9\n\nConclusions 18\n\nChecklist 20\n\nReferences 22\n',
   'text': '# Contents\n\nMental health of older persons: a human right and priority for sustainable development 2\n\nDeterminants of mental health in later life 2\n\nPrevalence of mental health conditions among older persons in the UNECE region 5\n\nPolicy strategies to improve the mental health of older persons 9\n\nConclusions 18\n\nChecklist 20\n\nReferences 22\n'},
  {'title': 'Suggested strategies',
   'node_id': '0001',
   'page_index': 1,
   's

In [14]:
llm_client = OpenAI(api_key=os.getenv("LLM_API_KEY"), base_url=os.getenv("LLM_BASE_URL"))

In [22]:
query = f"""
You are an expert question-answering assistant.

You only speak English.

Your task is to answer the user's question using ONLY the provided reference content.

The reference content comes from a hierarchical document indexing system (PageIndex). Each node represents a section of the original document and contains:

* node_id: unique identifier
* page_index: page index
* summary: summarized content of the node
* title: title of the chapter
* text: the content that the query is referencing to

Instructions:

1. Read the user's question carefully.
2. Search the reference nodes for relevant information.
3. Use the most relevant nodes to construct a concise, accurate, and complete answer.
4. Do NOT use external knowledge unless it is necessary to connect facts explicitly stated in the references.
5. If the answer cannot be found in the reference content, say:
   "The provided reference content does not contain enough information to answer this question."
6. Every factual statement in the answer must be supported by one or more citations.
7. Citations must use the node_id of the supporting node.
8. Multiple citations may be attached to a statement.
9. Prefer citations from the most specific node available.
10. Do not fabricate citations.

Output Format (JSON):

{{
"answer": "<final answer>",
"citations": [
{{
"node_id": "<node_id>",
"page_index": "<page_index>",
"title": "<title>",
"reason": "<why this node supports the answer>"
}}
]
}}

User Question:

What is the range of prevalence of depression in most countries? Do they have any significant differences?

Reference Content:

{result}
"""
messages = [{
    "role": "user",
    "content": query},
]

response = llm_client.chat.completions.create(
    model=os.getenv("LLM_MODEL_NAME"),
    messages=messages,
    extra_body={"thinking": {"type": "disabled"}},
    response_format={"type": "json_object"},
)

reasoning_content = response.choices[0].message.content
content = response.choices[0].message.content

print(content)
print("=" * 40)
print(response.model_dump_json(indent=2))

{
"answer": "In most countries, the prevalence of depression among persons aged 60 and older ranges from 20 to 35 percent. There are some significant differences, such as in Lithuania and Romania, where around 40 percent of older persons are affected.",
"citations": [
{
"node_id": "0013",
"page_index": "5",
"title": "Prevalence of common mental health conditions",
"reason": "Directly states the prevalence range and provides examples of significant differences."
}
]
}
{
  "id": "2400fafd-546e-404d-8ae6-1027db40ca89",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "{\n\"answer\": \"In most countries, the prevalence of depression among persons aged 60 and older ranges from 20 to 35 percent. There are some significant differences, such as in Lithuania and Romania, where around 40 percent of older persons are affected.\",\n\"citations\": [\n{\n\"node_id\": \"0013\",\n\"page_index\": \"5\",\n\"title\": \"Pre